:important: This is the notebook for my Wildfires project for SDS210. Broad structure as follows:

1. Import required packages
    

2. test pull data into project with the FIRMS API and then just use the bounding box for Australia.

3. reproject to correct CRS


4. 

In [1]:
import requests
import pandas as pd
import geopandas as gpd
import time
import datetime
import folium
from folium.plugins import MarkerCluster
import numpy as np

In [2]:
# We need to access the API and to do that, will use the map key that permits access.
MAP_KEY = '54684dde74a099b139ddbbef0f621891'

# Now let's check how many results we have

url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + MAP_KEY
try:
  test_response = requests.get(url)
  test_data = response.json()
  test_df = pd.Series(data)
  display(test_df)
except:
  # possible error, wrong MAP_KEY value, check for extra quotes, missing letters
  print ("There is an issue with the query. \nTry in your browser: %s" % url)

There is an issue with the query. 
Try in your browser: https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=54684dde74a099b139ddbbef0f621891


In [3]:
# this url will return information about all supported sensors and their corresponding datasets
# instead of 'all' you can specify individual sensor, ex:LANDSAT_NRT
sensor_data = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/' + MAP_KEY + '/all'
test_df = pd.read_csv(sensor_data)
display(test_df)

,data_id,min_date,max_date
0,MODIS_NRT,2026-03-01,2026-05-19
1,MODIS_SP,2000-11-01,2026-02-28
2,VIIRS_NOAA20_NRT,2026-04-01,2026-05-19
3,VIIRS_NOAA20_SP,2018-04-01,2026-03-31
4,VIIRS_NOAA21_NRT,2024-01-17,2026-05-19
5,VIIRS_SNPP_NRT,2026-04-01,2026-05-19
6,VIIRS_SNPP_SP,2012-01-20,2026-03-31
7,LANDSAT_NRT,2022-06-20,2026-05-18
8,GOES_NRT,2022-08-09,2026-05-19
9,BA_MODIS,2000-11-01,2026-02-01


In [4]:
# We are particularly interested in the wildfires in Australia and so will select this information using a bounding box. Aus = 110 -55, 180 -10
modis_nrt_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/MODIS_NRT/110,-50,160,-11/3'
modis_nrt_df = pd.read_csv(modis_nrt_url)

viirs_noaa20_nrt_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA20_NRT/110,-50,160,-11/3'
viirs_noaa20_nrt_df = pd.read_csv(viirs_noaa20_nrt_url)

viirs_noaa21_nrt_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA21_NRT/110,-50,160,-11/3'
viirs_noaa21_nrt_df = pd.read_csv(viirs_noaa21_nrt_url)

viirs_snpp_nrt_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_SNPP_NRT/110,-50,160,-11/3'
viirs_snpp_nrt_df = pd.read_csv(viirs_snpp_nrt_url)

In [5]:
print(modis_nrt_df.iloc[100,:])
print(viirs_noaa20_nrt_df.iloc[100,:])
print(viirs_noaa21_nrt_df.iloc[100,:])
print(viirs_snpp_nrt_df.iloc[100,:])

latitude       -37.96429
longitude      141.61533
brightness        365.05
scan                 1.0
track                1.0
acq_date      2026-05-18
acq_time             625
satellite           Aqua
instrument         MODIS
confidence           100
version           6.1NRT
bright_t31        292.25
frp               100.22
daynight               D
Name: 100, dtype: object
latitude       -14.82336
longitude      130.15909
bright_ti4        341.37
scan                0.49
track                0.4
acq_date      2026-05-17
acq_time             442
satellite            N20
instrument         VIIRS
confidence             n
version           2.0NRT
bright_ti5        296.28
frp                 7.46
daynight               D
Name: 100, dtype: object
latitude       -37.37374
longitude      141.07655
bright_ti4        327.51
scan                0.47
track               0.64
acq_date      2026-05-17
acq_time             525
satellite            N21
instrument         VIIRS
confidence             n


In [6]:
# Create a funtion that checks time of aquisition and calculates time delta
def add_time_since_acq(df):
    df["acq_date"] = pd.to_datetime(df["acq_date"])   # Ensuring date is in datetime format
    df["acq_datetime"] = pd.to_datetime(
        df["acq_date"].astype(str) + df["acq_time"].astype(str).str.zfill(4),    # Turns all values into 4 digit HHHH format
        format = "%Y-%m-%d%H%M"
        ).dt.tz_localize('UTC')

    current_time = pd.Timestamp.now('UTC')
    df["time_since_detection"] = current_time - df["acq_datetime"]

    # Calculate hours since detection
    df["hours_since_detection"] = df["time_since_detection"].dt.total_seconds()/3600

    # Convert data types to string format so folium can take them
    df["acq_date_str"] = df["time_since_detection"].astype(str)
    df["time_since_detection_str"] = df["acq_date"].astype(str)
    df["acq_datetime_str"] = df["acq_datetime"].astype(str)

    # Drop datetime columns and "time_since_detection_str" as it is covered by "hours_since_detection".
    df = df.drop(columns=["time_since_detection", "time_since_detection_str", "acq_date", "acq_datetime"])


    return df


modis_nrt_df = add_time_since_acq(modis_nrt_df)
viirs_noaa20_nrt_df = add_time_since_acq(viirs_noaa20_nrt_df)
viirs_noaa21_nrt_df = add_time_since_acq(viirs_noaa21_nrt_df)
viirs_snpp_nrt_df = add_time_since_acq(viirs_snpp_nrt_df)

In [7]:
# Convert to GeoDataFrame (WGS84)
modis_nrt_gdf = gpd.GeoDataFrame(
    modis_nrt_df, 
    geometry=gpd.points_from_xy(
        modis_nrt_df["longitude"], 
        modis_nrt_df["latitude"]
    ),
    crs="EPSG:4326")

viirs_noaa20_nrt_gdf = gpd.GeoDataFrame(
    viirs_noaa20_nrt_df, 
    geometry=gpd.points_from_xy(
        viirs_noaa20_nrt_df["longitude"], 
        viirs_noaa20_nrt_df["latitude"]
    ),
    crs="EPSG:4326")

viirs_noaa21_nrt_gdf = gpd.GeoDataFrame(
    viirs_noaa21_nrt_df, 
    geometry=gpd.points_from_xy(
        viirs_noaa21_nrt_df["longitude"], 
        viirs_noaa21_nrt_df["latitude"]
    ),
    crs="EPSG:4326")

viirs_snpp_nrt_gdf = gpd.GeoDataFrame(
    viirs_snpp_nrt_df, 
    geometry=gpd.points_from_xy(
        viirs_snpp_nrt_df["longitude"], 
        viirs_snpp_nrt_df["latitude"]
    ),
    crs="EPSG:4326")


print(f"Found {len(modis_nrt_gdf)} fire records.")
print(f"Found {len(viirs_noaa20_nrt_gdf)} fire records.")
print(f"Found {len(viirs_noaa21_nrt_gdf)} fire records.")
print(f"Found {len(viirs_snpp_nrt_gdf)} fire records.")

Found 335 fire records.
Found 1964 fire records.
Found 1877 fire records.
Found 1959 fire records.


In [9]:
# 1. Initialize the Folium map (The Control)
australia_bushfire_map = folium.Map(location=[-28.281828, 136.145401], zoom_start=5)

# 2. Add multiple datasets using GeoPandas (The Convenience)

modis_nrt_gdf.explore(
    m=australia_bushfire_map, 
    column='hours_since_detection', 
    name='MODIS NRT',
    tooltip=['confidence'], 
    cmap='YlOrRd_r',
    style_kwds={'fillOpacity': 0.4, 'color': 'white', 'weight': 0.1},
    show=True
)

viirs_noaa20_nrt_gdf.explore(
    m=australia_bushfire_map, 
    column='hours_since_detection', 
    name='VIIRS NOAA20 NRT',
    tooltip=['confidence'], 
    cmap='YlOrRd_r',
    style_kwds={'fillOpacity': 0.4, 'color': 'white', 'weight': 0.1},
    legend=False,
    show=True
)

viirs_noaa21_nrt_gdf.explore(
    m=australia_bushfire_map, 
    column='hours_since_detection', 
    name='VIIRS NOAA21 NRT',
    tooltip=['confidence'], 
    cmap='YlOrRd_r',
    style_kwds={'fillOpacity': 0.4, 'color': 'white', 'weight': 0.1},
    legend=False,
    show=True
)

viirs_snpp_nrt_gdf.explore(
    m=australia_bushfire_map, 
    column='hours_since_detection', 
    name='VIIRS SNPP NRT',
    tooltip=['confidence'], 
    cmap='YlOrRd_r',
    style_kwds={'fillOpacity': 0.4, 'color': 'white', 'weight': 0.1},
    legend=False,
    show=True
)


# 3. Add Folium plugins back on top
folium.LayerControl().add_to(australia_bushfire_map)

australia_bushfire_map

In [10]:
# Saving map for viewer access
australia_bushfire_map.save("australia_bushfire_map.html")